### 4. Entropy and OCV estimation

With experiment data in python structure format, we can use it to estimate entropy coefficient and OCV. 

We begin by importing packages:

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import os
import pickle

Defining cell data identification, temperatures and voltage plot limits 

In [ ]:
# Identifiers for each cell
cellIDs = ["BattMo"]

# Available temperatures for each cell
temps = [[45, 35, 25, 15]]

# Voltage limits for plotting
minV = [2.5]
maxV = [3.65]

Now we can call the estimation algorithm to estimate the entropy coefficient and OCV from desired datasets as follows:

In [ ]:
from Functions.process_function_OCV import processOCV  # previously converted function

for theID, dirname in enumerate(cellIDs):
    cellID = dirname
    # Remove trailing underscore if present
    if '_' in dirname:
        dirname = dirname.split('_')[0]

#--------------------uncoment line below for backupdata--------------------
    #OCVDir = f"DATA/{dirname}_OCV"
#--------------------------------------------------------------------------

#--------------------comment line below if  backup data is used------------
    OCVDir = f"../bdf-to-ds-Toolbox/out"
#--------------------------------------------------------------------------

    if not os.path.isdir(OCVDir):
        raise FileNotFoundError(
            f'Folder "{OCVDir}" not found in current folder.\n'
            f'Please change folders so that "{OCVDir}" is in the current folder and re-run.'
        )

    filetemps = np.array(temps[theID])
    numtemps = len(filetemps)
    data = [{} for _ in range(numtemps)]  # initialize data list

    # Load the OCV files
    for k, temp in enumerate(filetemps):

        if temp < 0:
            base = f"{cellID}_OCV_N{abs(temp):02d}"
        else:
            base = f"{cellID}_OCV_P{temp:02d}"

        file_s1 = os.path.join(OCVDir, f"{base}_S1.pkl")
        file_s3 = os.path.join(OCVDir, f"{base}_S3.pkl")

        if not os.path.isfile(file_s1):
            raise FileNotFoundError(f"Data file {file_s1} not found.")

        if not os.path.isfile(file_s3):
            raise FileNotFoundError(f"Data file {file_s3} not found.")

        with open(file_s1, 'rb') as f:
            script1_data = pickle.load(f)

        with open(file_s3, 'rb') as f:
            script3_data = pickle.load(f)

        data[k]['temp'] = temp
        data[k]['script1'] = script1_data
        data[k]['script3'] = script3_data

    # Call the processing function to generate OCV model
    model = processOCV(data, cellID, minV[theID], maxV[theID], savePlots=True)

    # Save model as a .pkl file
    os.makedirs('Models', exist_ok=True)
    model_filename = f"Models/{cellID}_model_ocv.pkl"
    with open(model_filename, 'wb') as f:
        pickle.dump(model, f)

    print(f"Processed model saved to {model_filename}")

Data is now procced and estimated entropy and OCV profiles can be compiled into tables and figures. This is done in the code cell bellow:

In [ ]:
# Create OCV table
OCVTable = pd.DataFrame({
    "SOC": model["SOCaprox"],
    "OCV": model["OCVaprox"]
})

# Create entropy (dU/dT) table
dUdT_Table = pd.DataFrame({
    "SOC": model["SOC"],
    "dUdT": model["OCVrel"]
})

#Export tables as .CSV files
os.makedirs('OCV_data', exist_ok=True) 
os.makedirs('Entropy_data', exist_ok=True)
OCVTable.to_csv(f"OCV_data/{cellID}_OCV.csv", index=False)   # Uncomment line to save OCV table
dUdT_Table.to_csv(f"Entropy_data/{cellID}_dUdT.csv", index=False) # Uncomment line to save Entropy profile table

#Ploting the entropy profile C/3

plt.figure()
plt.plot(model["SOC"], 1000 * model["OCVrel"])
plt.grid(True)

plt.title("Entropy figure from full-cell data", fontsize=16)
plt.xlabel("SOC [-]", fontsize=14)
plt.ylabel("dU/dT [mV/K]", fontsize=14)

plt.xlim([0, 1])
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

os.makedirs('Entropy_FIGURES', exist_ok=True)
plt.savefig(f'Entropy_FIGURES/{cellID}_Entropy_grid.png', dpi=300)

plt.show()



The estimated OCV or entropy coefficient data can be used to enhance the model cell voltage expressions. This concludes the session 4. Entropy and OCV estimation, in the next session we will present a code to do the same for half-cells.